# Imports

In [1]:
!pip install split-folders

In [2]:
import torch as torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
import torchvision.models as models
from torchvision import transforms
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import cv2
import os
import time
import random
import gc
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from PIL import Image
import scipy.stats as ss
import splitfolders

# utils.py

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Hyperparams
HEIGHT = 128
WIDTH = 128
CHANNELS = 6
FPS = 10
DURATION = 3
SEQ_LEN = FPS * DURATION

# MVO Prediction Logic Mapping
# FRONT: 0, LEFT: 1, RIGHT: 2
def get_label_id(label_name):
    mapping = {'front': 0, 'left': 1, 'right': 2}
    return mapping.get(label_name.lower(), 0)

# Intent files
VAL_POSITIONS = ''
TEST_POSITIONS = ''

# Contains both and only video and label directories
# Folder names are strictly "videos" and "labels"
DATA_DIR = r'/content/drive/MyDrive/Dataset_750'
CACHE_DIR = r'/content/drive/MyDrive/Dataset_cache'

VIDEO_DIR = os.path.join(DATA_DIR, "videos")
LABEL_DIR = os.path.join(DATA_DIR, "labels")

# Incomplete_EluSEEdate_Dataset
# Complete Final Training Datasets

# Check if paths exist to avoid errors later
if not os.path.exists(VIDEO_DIR):
    print(f"WARNING: Directory not found: {VIDEO_DIR}")
    print("Please update the VIDEO_DIR variable in this cell.")

if not os.path.exists(LABEL_DIR):
    print(f"WARNING: Directory not found: {LABEL_DIR}")
    print("Please update the LABEL_DIR variable in this cell.")

# preprocessor.py

In [5]:
def preprocess_and_cache_videos(video_folder, cache_folder, transforms, num_frames=30):
    """
    Pre-extract and cache video frames for faster training.

    Args:
        video_folder: Path to video files
        cache_folder: Path to save cached frames
        transforms: Torchvision transforms to apply
        num_frames: Number of frames per video (default: 30)
    """
    # Create cache directory if it doesn't exist
    os.makedirs(cache_folder, exist_ok=True)

    # Get all video files
    video_files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]

    print(f"Starting frame extraction for {len(video_files)} videos...")
    print(f"Cache directory: {cache_folder}")
    print(f"This may take a few minutes but only needs to be done once.\n")

    start_time = time.time()
    cached_count = 0
    skipped_count = 0

    for video_name in tqdm(video_files, desc="Preprocessing videos"):
        video_path = os.path.join(video_folder, video_name)
        cache_path = os.path.join(cache_folder, video_name.replace('.mp4', '.npy'))

        # Skip if already cached
        if os.path.exists(cache_path):
            skipped_count += 1
            continue

        # Extract frames from video
        cap = cv2.VideoCapture(video_path)
        frames = []

        for _ in range(num_frames):
            ret, frame = cap.read()
            if not ret:
                # Pad with black frame if video is shorter
                frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if transforms:
                    frame = Image.fromarray(frame)
                    frame_tensor = transforms(frame)
                else:
                    frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

            frames.append(frame_tensor)

        cap.release()

        # Stack and save as numpy array
        video_tensor = torch.stack(frames, dim=0)  # [30, 3, 128, 128]
        np.save(cache_path, video_tensor.numpy())

        cached_count += 1
        del frames, video_tensor

    elapsed_time = time.time() - start_time

    print(f"\n✓ Preprocessing completed!")
    print(f"  - Newly cached: {cached_count} videos")
    print(f"  - Already cached: {skipped_count} videos")
    print(f"  - Total time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    print(f"  - Cache size: ~{(cached_count + skipped_count) * 2:.0f} MB")
    print(f"\n✓ Ready for training! The dataset will now load from cache.")

preprocessing_transforms = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.ToTensor()
])
preprocess_and_cache_videos(VIDEO_DIR, CACHE_DIR, preprocessing_transforms)

Starting frame extraction for 750 videos...
Cache directory: /content/drive/MyDrive/Dataset_cache
This may take a few minutes but only needs to be done once.



Preprocessing videos:   0%|          | 0/750 [00:00<?, ?it/s]


✓ Preprocessing completed!
  - Newly cached: 0 videos
  - Already cached: 750 videos
  - Total time: 0.14 seconds (0.00 minutes)
  - Cache size: ~1500 MB

✓ Ready for training! The dataset will now load from cache.


# conv_lstm_classifier.py

In [6]:
class ConvLSTMCell(nn.Module):
    """
    The Single Memory Unit of the video.
    """

    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.kernel_size = kernel_size
        self.padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.bias = bias

        self.conv = nn.Conv2d(in_channels=self.input_dim + self.hidden_dim,
                              out_channels=4 * self.hidden_dim,
                              kernel_size=self.kernel_size,
                              padding=self.padding,
                              bias=self.bias)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state

        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)

        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

    def init_hidden(self, batch_size, image_size):
        height, width = image_size
        return (torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device),
                torch.zeros(batch_size, self.hidden_dim, height, width, device=self.conv.weight.device))

class ConvLSTM(nn.Module):
    """
    The Observer of the video
    """

    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers,
                 batch_first=True, bias=True, return_all_layers=False): # Changed default to batch_first=True
        super(ConvLSTM, self).__init__()

        self._check_kernel_size_consistency(kernel_size)
        kernel_size = self._extend_for_multilayer(kernel_size, num_layers)
        hidden_dim = self._extend_for_multilayer(hidden_dim, num_layers)

        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.num_layers = num_layers
        self.batch_first = batch_first
        self.bias = bias
        self.return_all_layers = return_all_layers

        cell_list = []
        for i in range(0, self.num_layers):
            cur_input_dim = self.input_dim if i == 0 else self.hidden_dim[i - 1]
            cell_list.append(ConvLSTMCell(input_dim=cur_input_dim,
                                          hidden_dim=self.hidden_dim[i],
                                          kernel_size=self.kernel_size[i],
                                          bias=self.bias))

        self.cell_list = nn.ModuleList(cell_list)

    def forward(self, input_tensor, hidden_state=None):
        # input_tensor format: [Batch, Time, Channel, Height, Width]

        if not self.batch_first:
            # (Time, Batch, Channel, Height, Width) -> (Batch, Time, Channel, Height, Width)
            input_tensor = input_tensor.permute(1, 0, 2, 3, 4)

        # Get Dimensions using the Correct Indices
        b, seq_len, _, h, w = input_tensor.size()

        if hidden_state is None:
            hidden_state = self._init_hidden(batch_size=b, image_size=(h, w))

        layer_output_list = []
        last_state_list = []

        cur_layer_input = input_tensor

        for layer_idx in range(self.num_layers):
            h, c = hidden_state[layer_idx]
            output_inner = []

            # Loop over TIME (seq_len), not Batch
            for t in range(seq_len):
                # Slice the time dimension: [Batch, Channel, H, W]
                # If layer_idx == 0, cur_layer_input is [B, T, C, H, W]
                # If layer_idx > 0, cur_layer_input is [B, T, Hidden, H, W] (from previous layer stack)

                input_t = cur_layer_input[:, t, :, :, :]

                h, c = self.cell_list[layer_idx](input_tensor=input_t, cur_state=[h, c])
                output_inner.append(h)

            # Stack along Time dimension (dim=1 because we enforce batch_first internally now)
            layer_output = torch.stack(output_inner, dim=1)
            cur_layer_input = layer_output

            layer_output_list.append(layer_output)
            last_state_list.append([h, c])

        if not self.return_all_layers:
            layer_output_list = layer_output_list[-1:]
            last_state_list = last_state_list[-1:]

        return layer_output_list, last_state_list

    def _init_hidden(self, batch_size, image_size):
        init_states = []
        for i in range(self.num_layers):
            init_states.append(self.cell_list[i].init_hidden(batch_size, image_size))
        return init_states

    @staticmethod
    def _check_kernel_size_consistency(kernel_size):
        if not (isinstance(kernel_size, tuple) or
                (isinstance(kernel_size, list) and all([isinstance(elem, tuple) for elem in kernel_size]))):
            raise ValueError('`kernel_size` must be tuple or list of tuples')

    @staticmethod
    def _extend_for_multilayer(param, num_layers):
        if not isinstance(param, list):
            param = [param] * num_layers
        return param


class ConvLSTMModel(nn.Module):
    """
    The Judge of the video.
    """
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, height, width,
                 batch_first=True, bias=True, return_all_layers=False, num_classes=3, dropout_rate=0.5):
        super(ConvLSTMModel, self).__init__()

        # Ensure batch_first is passed correctly
        self.convlstm = ConvLSTM(input_dim, hidden_dim, kernel_size, num_layers,
                                 batch_first=batch_first, bias=bias,
                                 return_all_layers=return_all_layers)

        # Dropout for regularization (prevents overfitting)
        self.dropout = nn.Dropout(p=dropout_rate)

        # Input to linear is: Hidden_Dim * H * W
        self.linear = nn.Linear(hidden_dim[-1] * height * width, num_classes)

    def forward(self, input_tensor, hidden_state=None):
        x, _ = self.convlstm(input_tensor)

        # x[0] shape is now guaranteed to be [Batch, Time, Hidden, H, W]
        # We take the last time step: x[0][:, -1, :, :, :]

        last_time_step = x[0][:, -1, :, :, :]

        # Flatten starting from dimension 1 (Channels/Hidden)
        flattened = torch.flatten(last_time_step, start_dim=1)

        # Apply dropout for regularization (only active during training)
        flattened = self.dropout(flattened)

        output = self.linear(flattened)
        return output

# dataset.py

In [7]:
class MVOVideoDataset(Dataset):
    """
    This takes a 3-second video and turns it into
    a 'data packet' for the AI to study.

    Now with frame caching support for 10-20x faster data loading!
    """
    def __init__(self, video_folder, label_folder, transforms=None, cache_folder=None):
        self.video_folder = video_folder
        self.label_folder = label_folder
        self.cache_folder = cache_folder
        self.transforms = transforms
        self.use_cache = False

        valid_video_files = []
        valid_csv_files = []
        try:
            all_video_files = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]
            for video_name in all_video_files:
                video_path = os.path.join(video_folder, video_name)

                # Check for corresponding CSV file (same name, just .csv instead of .mp4)
                csv_name = video_name.replace('.mp4', '.csv')
                csv_path = os.path.join(label_folder, csv_name)

                if os.path.exists(video_path) and os.path.exists(csv_path):
                    valid_video_files.append(video_name)
                    valid_csv_files.append(csv_name)
                else:
                    if not os.path.exists(video_path):
                        print(f"WARNING: Video file not found: {video_path}. Skipping.")
                    if not os.path.exists(csv_path):
                        print(f"WARNING: Label file not found for video {video_name}. Skipping.")

            self.video_files = valid_video_files
            self.csv_files = valid_csv_files

            # Check if cache is available
            if cache_folder and os.path.exists(cache_folder):
                cached_count = sum(1 for f in self.video_files
                                  if os.path.exists(os.path.join(cache_folder, f.replace('.mp4', '.npy'))))
                if cached_count > 0:
                    self.use_cache = True
                    print(f"✓ Using cached frames from: {cache_folder}")
                    print(f"  ({cached_count}/{len(self.video_files)} videos cached)")

            if not self.use_cache:
                print(f"Loading frames from videos (no cache available)")

            print(f"Initialized dataset with {len(self.video_files)} valid video-label pairs.")

        except FileNotFoundError:
            print(f"Error: Could not find folder {video_folder} or {label_folder}.")
            self.video_files = []

        self.split_type = ''
        self.positions = [] # If split_type == 'train', this would not be filled.

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_name = self.video_files[idx]

        # Get the Label from the matching CSV file
        csv_name = video_name.replace('.mp4', '.csv')
        csv_path = os.path.join(self.label_folder, csv_name)

        df = pd.read_csv(csv_path)
        label = self.labeler(df)

        if self.split_type == 'TRAIN':
            intent_position = self.get_intent_position()
        else:
            intent_position = self.positions[idx]

        intent = self.get_intent(intent_position, df)

        # Load from cache if available (FAST PATH)
        if self.use_cache:
            cache_path = os.path.join(self.cache_folder, video_name.replace('.mp4', '.npy'))
            if os.path.exists(cache_path):
                # Load pre-processed frames from cache (~5-10ms)
                video_tensor = torch.from_numpy(np.load(cache_path))

                # Split video tensor to frames
                frames = []
                for i in range(video_tensor.shape[0]):
                    frame_tensor = video_tensor[i]
                    frame_tensor = self.get_intent_torch(intent_position, i, intent, frame_tensor)
                    frames.append(frame_tensor)

                video_tensor = torch.stack(frames, dim=0)
            else:
                # Fallback to video decoding if cache missing for this video
                video_tensor = self._load_from_video(video_name, intent, intent_position)
        else:
            # Load from video file (SLOW PATH - ~100-200ms)
            video_tensor = self._load_from_video(video_name, intent, intent_position)

        return video_tensor, torch.tensor(label).long()

    def _load_from_video(self, video_name, intent, intent_position):
        """Load and process frames from video file (used when cache is not available)"""
        video_path = os.path.join(self.video_folder, video_name)

        cap = cv2.VideoCapture(video_path)
        frames = []
        for i in range(30):
            ret, frame = cap.read()
            if not ret:
                # If a video is shorter than 3s, pad with a black frame
                frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transforms:
                    frame = Image.fromarray(frame)
                    frame_tensor = self.transforms(frame)
                else:
                    # Fallback if no transforms are provided
                    frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

            frame_tensor = self.get_intent_torch(intent_position, i, intent, frame_tensor)

            frames.append(frame_tensor)

        cap.release()

        # Convert list to tensor [30, 3, 128, 128]
        video_tensor = torch.stack(frames, dim=0)

        # Memory cleanup
        del frames

        return video_tensor

    def get_intent_torch(self, intent_position, i, intent, frame_tensor):
        # Create a tensor for the intent with the same spatial dimensions as the video frames
        # Used for no intent
        intent_torch = torch.zeros((3, HEIGHT, WIDTH))
        # If intent exists, add intent in its intent position for 1 second (10 frames)
        if intent_position != -1 and intent_position <= i and (intent_position + 10) > i:
            # Convert intent to one-hot vector by filling the specified channel with 1
            intent_torch[intent, :, :] = 1

        # Append the intent as a channel to the video frame
        frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)

        return frame_tensor

    def labeler(self, df):
        df_lbl_count = []
        # Logic to handle counts for classes 0, 1, and 2
        # Added a check for the column name to prevent KeyErrors
        col = 'label_id_corrected' if 'label_id_corrected' in df.columns else df.columns[-1]
        counts = df[col].tail(24).value_counts()

        for i in range(0, 3):
            df_lbl_count.append(counts.get(i, 0))

        if df_lbl_count[0] == 24:
            label = 0 # Front
        elif df_lbl_count[1] > df_lbl_count[2]:
            label = 1 # Left
        elif df_lbl_count[1] < df_lbl_count[2]:
            label = 2 # Right
        else:
            label = df[col].tail(12).mode()[0]

        return label

    def get_intent_position(self):
        # 50% of the dataset have intent
        if random.random() < 0.6:
            # The time positions of the first 2 seconds (videos - 10 fps)
            start_frame = 0
            end_frame = 20
            median = (start_frame + end_frame)/2
            range_zero = np.arange(-median, median)

            # Obtain the probability of selecting a timestamp using the adjacent 0.5 areas
            smaller_range = range_zero - 0.5
            higher_range = range_zero + 0.5

            # Probability is the difference of the probability of higher range and lower range
            probability = ss.norm.cdf(higher_range) - ss.norm.cdf(smaller_range)

            # Normalize the probabilities
            # Each probability in probability range is divided by the sum of the probabilities in probability range
            probability /= probability.sum()

            # Select a timestamp based on the probabilities
            range = np.arange(start_frame, end_frame)
            intent_position = np.random.choice(range, p=probability)
        else:
            intent_position = -1

        return intent_position

    def get_intent(self, intent_position, df):
        # Check if the data has no intent
        if intent_position != -1:
            intent = self.labeler(df)
        else:
            intent = -1
        return intent

    def class_counter(self):
        label_counts = {0: 0, 1: 0, 2: 0}

        for csv_file in self.csv_files:
            csv_path = os.path.join(self.label_folder, csv_file)
            df = pd.read_csv(csv_path)
            label = self.labeler(df)
            label_counts[label] += 1

        return label_counts, sum(label_counts.values())

    def set_split_type(self, type, len_dataset):
        global VAL_POSITIONS
        global TEST_POSITIONS
        self.split_type = type

        if self.split_type == 'VALIDATION':
            if VAL_POSITIONS == '':
                for _ in range(len_dataset):
                    self.positions.append(self.get_intent_position())
                np.save('val_intent_positions.npy', np.array(self.positions))
                VAL_POSITIONS = 'val_intent_positions.npy'
            else:
                self.positions = list(np.load(VAL_POSITIONS))
        elif self.split_type == 'TEST':
            if TEST_POSITIONS == '':
                for _ in range(len_dataset):
                    self.positions.append(self.get_intent_position())
                np.save('test_intent_positions.npy', np.array(self.positions))
                TEST_POSITIONS = 'test_intent_positions.npy'
            else:
                self.positions = list(np.load(TEST_POSITIONS))

        return ''

# train.py

In [ ]:
# Configurations
BATCH = 2  # Reduced from 5 to handle larger dataset (500 videos)
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
SAVED_MODEL_PATH = "best_convlstm.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Gradient Accumulation Configuration
ACCUMULATION_STEPS = 4  # Effective batch size = BATCH * ACCUMULATION_STEPS (2 * 4 = 8)

# Early Stopping Configuration
EARLY_STOP_PATIENCE = 2  # Stop if no improvement for 5 epochs
MIN_DELTA = 0.01  # Minimum change to qualify as improvement (0.01%)

# Setting a fixed random seed to ensure that
# we get the exact same data split every time we run the script
SEED = 8
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Model Parameters
PARAMS = {
    'input_dim': 6,
    'hidden_dim': [64, 32],
    'kernel_size': (3, 3),
    'num_layers': 2,
    'height': HEIGHT,
    'width': WIDTH,
    'num_classes': 3
}

def train_one_epoch(model, loader, criterion, optimizer, accumulation_steps=1, max_grad_norm=1.0):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    total_grad_norm = 0.0
    optimizer.zero_grad()  # Zero gradients at the start

    loop = tqdm(loader, leave=True)
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.float().to(DEVICE)
        targets = targets.to(DEVICE)

        # Forward
        scores = model(data)
        loss = criterion(scores, targets)

        # Scale loss by accumulation steps (important for correct gradient magnitude)
        loss = loss / accumulation_steps

        # Backward (accumulate gradients)
        loss.backward()

        # Only update weights every accumulation_steps batches
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            # Gradient Clipping: Prevent exploding gradients in ConvLSTM
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            total_grad_norm += grad_norm.item()

            optimizer.step()
            optimizer.zero_grad()

            # Update progress bar on weight updates
            loop.set_description(f"Loss: {loss.item()*accumulation_steps:.4f} | GradNorm: {grad_norm.item():.3f} | Step: {(batch_idx+1)//accumulation_steps}")

        # Stats (track unscaled loss for reporting)
        running_loss += loss.item() * accumulation_steps
        _, predictions = scores.max(1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

        # Memory cleanup: Delete intermediate tensors
        del data, targets, scores, loss, predictions

    # Clear GPU cache after epoch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # Average gradient norm per weight update (not per batch)
    num_updates = (len(loader) + accumulation_steps - 1) // accumulation_steps
    avg_grad_norm = total_grad_norm / num_updates
    return running_loss / len(loader), 100 * correct / total, avg_grad_norm

def main():
    # Data Setup
    transforms_train = transforms.Compose([
        transforms.Resize((HEIGHT, WIDTH)),
        transforms.ToTensor()
    ])

    train_dir = os.path.join("output", "train")
    val_dir = os.path.join("output", "val")
    test_dir = os.path.join("output", "test")

    train_dir_vid = os.path.join(train_dir, "videos")
    val_dir_vid = os.path.join(val_dir, "videos")
    test_dir_vid = os.path.join(test_dir, "videos")

    train_lbl_vid = os.path.join(train_dir, "labels")
    val_lbl_vid = os.path.join(val_dir, "labels")
    test_lbl_vid = os.path.join(test_dir, "labels")

    # Split dataset if have not yet split
    if os.path.exists("output") and len(os.listdir(VIDEO_DIR)) == 0 and len(os.listdir(LABEL_DIR)) == 0:
        # Move the files from the input directory into three directories: training (80%), validation (20%), and testing (20%)
        splitfolders.ratio(DATA_DIR, output="output", seed=8, ratio=(.6, .2, .2), group="sibling",move=True, shuffle=True)

    train_dataset = MVOVideoDataset(train_dir_vid, train_lbl_vid, transforms=transforms_train, cache_folder=CACHE_DIR)
    val_dataset = MVOVideoDataset(val_dir_vid, val_lbl_vid, transforms=transforms_train, cache_folder=CACHE_DIR)
    test_dataset = MVOVideoDataset(test_dir_vid, test_lbl_vid, transforms=transforms_train, cache_folder=CACHE_DIR)

    print(f"Data Split -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test (Unused): {len(test_dataset)}")

    train_dataset.set_split_type('TRAIN', len(train_dataset))
    val_dataset.set_split_type('VALIDATION', len(val_dataset))

    train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH, shuffle=False, num_workers=0)

    # Calculate class instances for class weights
    label_counts, total_count = train_dataset.class_counter()

    # Add 1 to avoid division by zero
    front_weight = total_count / (label_counts[0] + 1)
    left_weight = total_count / (label_counts[1] + 1)
    right_weight = total_count / (label_counts[2] + 1)

    print(f"Front class instances: {label_counts[0]} -> Front weight: {front_weight}")
    print(f"Left class instances: {label_counts[1]} -> Left weight: {left_weight}")
    print(f"Right class instances: {label_counts[2]} -> Right weight: {right_weight}")

    # Model Setup
    model = ConvLSTMModel(
        input_dim=PARAMS['input_dim'],
        hidden_dim=PARAMS['hidden_dim'],
        kernel_size=PARAMS['kernel_size'],
        num_layers=PARAMS['num_layers'],
        height=PARAMS['height'],
        width=PARAMS['width'],
        num_classes=PARAMS['num_classes']
    ).to(DEVICE)

    # CrossEntropyLoss is used to handle class imbalance
    class_weights = torch.FloatTensor([front_weight,left_weight,right_weight]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # Learning Rate Scheduler: Reduces LR when validation accuracy plateaus
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',           # Maximize validation accuracy
        factor=0.5,           # Reduce LR by half
        patience=3,           # Wait 3 epochs before reducing
        min_lr=1e-7           # Don't go below this LR
    )

    # Training Loop with Early Stopping
    best_acc = 0
    epochs_no_improve = 0
    early_stop = False
    print(f"Training on {DEVICE} with {len(train_dataset)} videos.")
    print(f"Gradient Accumulation: {ACCUMULATION_STEPS} steps (Effective batch size: {BATCH * ACCUMULATION_STEPS})")
    print(f"Early stopping enabled: patience={EARLY_STOP_PATIENCE}, min_delta={MIN_DELTA}%")

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        train_loss, train_acc, avg_grad_norm = train_one_epoch(
            model, train_loader, criterion, optimizer,
            accumulation_steps=ACCUMULATION_STEPS, max_grad_norm=1.0
        )

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_latencies = []

        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(val_loader):
                x = x.float().to(DEVICE)
                y = y.to(DEVICE)

                # Measure inference time
                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()

                start_time = time.perf_counter()
                scores = model(x)

                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()

                end_time = time.perf_counter()

                # Skip first batch for warm-up
                if batch_idx >= 1:
                    val_latencies.append(end_time - start_time)

                _, preds = scores.max(1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

                # Memory cleanup: Delete intermediate tensors
                del x, y, scores, preds

        # Clear GPU cache after validation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        val_acc = 100 * val_correct / val_total
        avg_val_latency_ms = (np.mean(val_latencies) * 1000) if len(val_latencies) > 0 else 0
        val_throughput = (1 / np.mean(val_latencies)) if len(val_latencies) > 0 else 0

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
        print(f"Avg Gradient Norm: {avg_grad_norm:.4f} (clipped at 1.0)")
        print(f"Val Inference: {avg_val_latency_ms:.2f} ms/batch | {val_throughput:.2f} batches/sec")
        print(f"Current LR: {optimizer.param_groups[0]['lr']:.2e}")

        # Update learning rate based on validation accuracy
        scheduler.step(val_acc)

        # Save Best Model & Early Stopping Check
        if val_acc > best_acc + MIN_DELTA:
            best_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), SAVED_MODEL_PATH)
            print(f"✓ New best model saved! ({val_acc:.2f}%)")
        else:
            epochs_no_improve += 1
            print(f"No improvement for {epochs_no_improve} epoch(s). Best: {best_acc:.2f}%")

            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"\n⚠ Early stopping triggered! No improvement for {EARLY_STOP_PATIENCE} epochs.")
                print(f"Best validation accuracy: {best_acc:.2f}%")
                early_stop = True
                break

    if not early_stop:
        print(f"\n✓ Training completed all {NUM_EPOCHS} epochs.")
    print(f"Final best validation accuracy: {best_acc:.2f}%")

if __name__ == "__main__":
    main()

Copying files: 750 files [15:51,  1.27s/ files]


✓ Using cached frames from: /content/drive/MyDrive/Dataset_cache
  (450/450 videos cached)
Initialized dataset with 450 valid video-label pairs.
✓ Using cached frames from: /content/drive/MyDrive/Dataset_cache
  (150/150 videos cached)
Initialized dataset with 150 valid video-label pairs.
✓ Using cached frames from: /content/drive/MyDrive/Dataset_cache
  (150/150 videos cached)
Initialized dataset with 150 valid video-label pairs.
Data Split -> Train: 450 | Val: 150 | Test (Unused): 150
Front class instances: 204 -> Front weight: 2.1951219512195124
Left class instances: 133 -> Left weight: 3.3582089552238807
Right class instances: 113 -> Right weight: 3.9473684210526314
Training on cuda with 450 videos.
Gradient Accumulation: 4 steps (Effective batch size: 8)
Early stopping enabled: patience=2, min_delta=0.01%

Epoch 1/1


  0%|          | 0/225 [00:00<?, ?it/s]

Train Loss: 1.0760 | Train Acc: 44.67% | Val Acc: 54.67%
Avg Gradient Norm: 4.5414 (clipped at 1.0)
Val Inference: 202.62 ms/batch | 4.94 batches/sec
Current LR: 1.00e-04
✓ New best model saved! (54.67%)

✓ Training completed all 1 epochs.
Final best validation accuracy: 54.67%


# tester.py

In [ ]:
class Tester:
    """
    It loads a trained model, feeds it unseen data,
    and records how accurately and how fast the model makes decisions.
    """
    def __init__(self, model_path, device):
        self.model_path = model_path
        self.device = device
        self.transforms = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((HEIGHT, WIDTH))
        ])

        # Load the Architecture
        self.model = ConvLSTMModel(
            input_dim=6,
            hidden_dim=[64, 32],
            kernel_size=(3, 3),
            num_layers=2,
            height=HEIGHT,
            width=WIDTH,
            num_classes=3,
            dropout_rate=0.5  # Dropout not applied during eval mode
        ).to(self.device)

        # Load the Weights
        self.load_weights()

    def load_weights(self):
        # Attempts to load the best model file and sets it to evaluation mode
        if os.path.exists(self.model_path):
            print(f"Loading model from {self.model_path}...")
            self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
            self.model.eval()
        else:
            raise FileNotFoundError(f"Model file not found at {self.model_path}. Did you run train.py?")

    def test(self):
        # The main evaluation loop.
        print("Preparing Test Data...")

        test_dir = os.path.join("output", "test")
        test_dir_vid = os.path.join(test_dir, "videos")
        test_lbl_vid = os.path.join(test_dir, "labels")
        test_dataset = MVOVideoDataset(test_dir_vid, test_lbl_vid, transforms=self.transforms)
        test_dataset.set_split_type('TEST', len(test_dataset))

        test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0, cache_folder=CACHE_DIR)

        print(f"Evaluating {len(test_dataset)} videos and measuring latency...")

        all_preds = []
        all_labels = []
        latencies = []

        # Evaluation Loop
        with torch.no_grad():
            for i, (video_tensor, labels) in enumerate(tqdm(test_loader, leave=True)):
                video_tensor = video_tensor.float().to(self.device)
                labels = labels.to(self.device)

                # Latency Measurement Start
                if self.device.type == 'cuda':
                    torch.cuda.synchronize()

                start_time = time.perf_counter() # Timer

                outputs = self.model(video_tensor) # Forward pass (The Inference)

                if self.device.type == 'cuda':
                    torch.cuda.synchronize() # Wait for the GPU to finish the math

                end_time = time.perf_counter()
                # Latency Measurement End

                # We skip the first 5 frames ('warm-up')
                if i >= 5:
                    latencies.append(end_time - start_time)

                # Convert raw scores to the predicted class index (0, 1, or 2)
                _, predicted = torch.max(outputs.data, 1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                # Memory cleanup: Delete intermediate tensors
                del video_tensor, labels, outputs, predicted

                # Periodic cache clearing every 50 videos
                if i % 50 == 0:
                    if self.device.type == 'cuda':
                        torch.cuda.empty_cache()
                    gc.collect()

        # Calculate Latency Stats
        avg_latency_ms = np.mean(latencies) * 1000 if len(latencies) > 0 else 0
        inf_fps = 1 / np.mean(latencies) if len(latencies) > 0 else 0

        # Calculate and Print all results
        self.calculate_metrics(all_labels, all_preds, avg_latency_ms, inf_fps)

        # Save detailed logs to a CSV
        self.save_results(all_labels, all_preds)

    def calculate_metrics(self, y_true, y_pred, avg_latency_ms, inf_fps):
        # Computes statistical performance and prints the Final Report
        print(f"Avg Latency:        {avg_latency_ms:.2f} ms per video clip")
        print(f"Inference Speed:    {inf_fps:.2f} clips per second")
        # Computes statistical performance and prints the Final Report
        print("\n" + "-"*40)
        print("       FINAL PERFORMANCE REPORT       ")
        print("-"*40)

        # Accuracy
        acc = accuracy_score(y_true, y_pred)
        print(f"Overall Accuracy:   {acc*100:.2f}%")

        # Precision and Recall
        precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)

        print(f"Precision:          {precision:.4f}")
        print(f"Recall:             {recall:.4f}")
        print("-" * 40)

        print(f"Avg Latency:        {avg_latency_ms:.2f} ms per video clip")
        print(f"Inference Speed:    {inf_fps:.2f} clips per second")

        print("-" * 40)
        print("Detailed Class Report:")
        # Generates a table for Front(0), Left(1), Right(2)
        print(classification_report(y_true, y_pred, target_names=['Front', 'Left', 'Right'], zero_division=0))

    def save_results(self, y_true, y_pred):
        # Creates a CSV to see exactly which videos failed
        df = pd.DataFrame({
            'Actual_Label': y_true,
            'Predicted_Label': y_pred
        })

        # Map numbers back to words for readability
        label_map = {0: 'Front', 1: 'Left', 2: 'Right'}
        df['Actual_Text'] = df['Actual_Label'].map(label_map)
        df['Predicted_Text'] = df['Predicted_Label'].map(label_map)

        # Check if correct
        df['Correct'] = df['Actual_Label'] == df['Predicted_Label']

        save_path = "test_results.csv"
        df.to_csv(save_path, index=False)
        print(f"\nDetailed predictions saved to '{save_path}'")

if __name__ == "__main__":
    # Configuration
    MODEL_PATH = "best_convlstm.pth"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Run the Tester
    tester = Tester(MODEL_PATH, DEVICE)
    tester.test()

Loading model from best_convlstm.pth...
Preparing Test Data...
Loading frames from videos (no cache available)
Initialized dataset with 150 valid video-label pairs.
Evaluating 150 videos and measuring latency...


  0%|          | 0/150 [00:00<?, ?it/s]

Avg Latency:        68.59 ms per video clip
Inference Speed:    14.58 clips per second

----------------------------------------
       FINAL PERFORMANCE REPORT       
----------------------------------------
Overall Accuracy:   55.33%
Precision:          0.3996
Recall:             0.5533
----------------------------------------
Avg Latency:        68.59 ms per video clip
Inference Speed:    14.58 clips per second
----------------------------------------
Detailed Class Report:
              precision    recall  f1-score   support

       Front       0.56      0.69      0.62        64
        Left       0.54      0.89      0.67        44
       Right       0.00      0.00      0.00        42

    accuracy                           0.55       150
   macro avg       0.37      0.52      0.43       150
weighted avg       0.40      0.55      0.46       150


Detailed predictions saved to 'test_results.csv'
